In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    regexp_extract,
    regexp_replace,
    parse_json,
    lit
)

In [0]:

base_path = "dbfs:/Volumes/iics/casemix/history/json_test/extracted/carereport_hsar_20250305_162803/"

df = (
    spark.read.format("json").option("multiline", "true").load(base_path)
    .withColumn("file_path", lit(base_path))
    .withColumn("file_name", regexp_replace(col("file_path"), base_path, ""))
    .withColumn("loaded_at", current_timestamp())

)



In [0]:
display(df)

In [0]:
df_variant = df.select(
    "file_path",
    "file_name",
    parse_json("json_text").alias("json_data"),
    "loaded_at"
)


In [0]:
df_variant.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("iics.casemix.json_raw_test")

In [0]:
%sql
SELECT * FROM iics.casemix.json_raw_test

In [0]:
print("hi")

In [0]:
(
    spark.readStream .format("cloudFiles") 
    .option("cloudFiles.format", "json") 
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") 
    .load("/Volumes/iics/casemix/history/json_test/extracted/carereport_hsar_20250305_162803/") 
    .writeStream.option("mergeSchema", "true")
    .toTable("iics.casemix.json_raw_test1") 

)

In [0]:
(
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option(
            "cloudFiles.schemaLocation",
            "/Volumes/iics/casemix/history/json_test/schema/carereport_hsar"
        )
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load("/Volumes/iics/casemix/history/json_test/extracted/carereport_hsar_20250305_162803/")
        .writeStream
        .option(
            "checkpointLocation",
            "/Volumes/iics/casemix/history/json_test/checkpoint/carereport_hsar"
        )
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable("iics.casemix.json_raw_test1")
)

In [0]:


(
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option(
            "cloudFiles.schemaLocation",
            "/Volumes/iics/casemix/history/json_test/schema/carereport_hsar"
        )
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load("/Volumes/iics/casemix/history/json_test/extracted/carereport_hsar_20250305_162803/")
        .withColumn("file_path", col("_metadata.file_path"))
        .withColumn(
            "file_name",
            regexp_extract(col("_metadata.file_path"), r"([^/]+)$", 1)
        )
        .withColumn("loaded_at", current_timestamp())
        .writeStream
        .option(
            "checkpointLocation",
            "/Volumes/iics/casemix/history/json_test/checkpoint/carereport_hsar"
        )
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable("iics.casemix.json_raw_test1")
)

In [0]:
%sql
-- DROP TABLE IF EXISTS iics.casemix.json_raw_test1

In [0]:
%sql
SELECT * FROM iics.casemix.json_raw_test

In [0]:
%sql
SELECT * FROM iics.casemix.json_raw_test1